In [1]:
import numpy as np
import random

In [2]:
# ==========================================
# 1. ENVIRONMENT SETUP
# ==========================================
GRID_SIZE = 3
START_STATE = (0, 0)
GOAL_STATE = (2, 2)  # Reward: +10
HOLE_STATE = (1, 1)  # Reward: -10 (Terminal state)

# Actions: 0=Up, 1=Right, 2=Down, 3=Left
NUM_ACTIONS = 4

q_table = np.zeros((GRID_SIZE, GRID_SIZE, NUM_ACTIONS))

In [3]:
# ==========================================
# 2. HYPERPARAMETERS
# ==========================================
ALPHA = 0.1     # Learning Rate
GAMMA = 0.9     # Discount Factor
EPSILON = 0.1   # Exploration Rate
EPISODES = 10000 # Increased because slipping makes learning harder!

In [4]:
# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def take_action(state, intended_action):
    """
    Applies the slippery 'Frozen Lake' physics.
    80% chance to move in the intended direction.
    10% chance to slip 90 degrees to the left.
    10% chance to slip 90 degrees to the right.
    """
    roll = random.uniform(0, 1)
    
    if roll < 0.80:
        actual_action = intended_action
    elif roll < 0.90:
        # Slip counter-clockwise (e.g., intended Up(0) -> slips Left(3))
        actual_action = (intended_action - 1) % 4
    else:
        # Slip clockwise (e.g., intended Up(0) -> slips Right(1))
        actual_action = (intended_action + 1) % 4

    # Now execute the actual movement
    r, c = state
    if actual_action == 0 and r > 0:               r -= 1  # Move Up
    elif actual_action == 1 and c < GRID_SIZE - 1: c += 1  # Move Right
    elif actual_action == 2 and r < GRID_SIZE - 1: r += 1  # Move Down
    elif actual_action == 3 and c > 0:             c -= 1  # Move Left
    
    return (r, c)

def get_reward(state):
    """Returns the reward for stepping into a state."""
    if state == GOAL_STATE: return 10
    if state == HOLE_STATE: return -10
    return -1


In [5]:
# ==========================================
# 4. THE Q-LEARNING ALGORITHM
# ==========================================
for episode in range(EPISODES):
    state = START_STATE
    
    # Loop ends if agent reaches the Goal OR falls in the Hole
    while state != GOAL_STATE and state != HOLE_STATE:
        
        # A. Choose action (Epsilon-Greedy)
        if random.uniform(0, 1) < EPSILON:
            action = random.choice([0, 1, 2, 3])
        else:
            action = np.argmax(q_table[state[0], state[1]])
            
        # B. Take the step (The environment applies the slip physics here!)
        next_state = take_action(state, action)
        reward = get_reward(next_state)
        
        # C. Q-Table Update (Bootstrapping)
        old_value = q_table[state[0], state[1], action]
        
        # If the next state is game over (goal or hole), there are no future rewards
        if next_state == GOAL_STATE or next_state == HOLE_STATE:
            next_max = 0
        else:
            next_max = np.max(q_table[next_state[0], next_state[1]])
            
        # Bellman Equation
        new_value = old_value + ALPHA * (reward + (GAMMA * next_max) - old_value)
        q_table[state[0], state[1], action] = new_value
        
        # D. Move to next state
        state = next_state


In [6]:
# ==========================================
# 5. VISUALIZING THE RESULTS
# ==========================================
print("Training Complete! Here is the Agent's Map:")
print("-" * 50)

actions_symbols = ['↑', '→', '↓', '←']

for r in range(GRID_SIZE):
    row_visual = []
    for c in range(GRID_SIZE):
        if (r, c) == GOAL_STATE:
            row_visual.append("[  GOAL  ]")
        elif (r, c) == HOLE_STATE:
            row_visual.append("[  HOLE  ]")
        else:
            best_action = np.argmax(q_table[r, c])
            best_value = np.max(q_table[r, c])
            row_visual.append(f"[{actions_symbols[best_action]}: {best_value:>+5.1f}]")
            
    print("  ".join(row_visual))
    print("")

Training Complete! Here is the Agent's Map:
--------------------------------------------------
[→:  +0.5]  [→:  +1.6]  [↓:  +3.8]

[↓:  +1.8]  [  HOLE  ]  [↓:  +8.0]

[→:  +4.8]  [→:  +6.7]  [  GOAL  ]

